In [1]:
# %% Shared setup
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Tapiculin"
STUDENT_ID = "TUPM-26-1932"
id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

X = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)          # measurement matrix
labels = np.loadtxt(DATA_DIR / "secom_labels.data", usecols=0, dtype=int)
assert len(labels) == len(X)
feature_names = [f"Feature_{i:03d}" for i in range(X.shape[1])]

missing = np.isnan(X)
observed_n = (~missing).sum(axis=0)
obs_min = np.where(missing, np.inf, X).min(axis=0)
obs_max = np.where(missing, -np.inf, X).max(axis=0)
all_missing = observed_n == 0
constant = (observed_n > 0) & (obs_min == obs_max)
usable = np.flatnonzero(~(all_missing | constant))               # at least two distinct observed values
print("Records:", X.shape[0], "| Features:", X.shape[1], "| Usable features:", len(usable))
print("Seed digit:", SEED_DIGIT, "| ID sum:", ID_SUM, "| Random seed:", RANDOM_SEED)


# %% EXERCISE 2 - Grouped Outcome Comparison (Pandas)
df = pd.DataFrame(X, columns=feature_names)
df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")
df.index.name = "Example_ID"                                          # original SECOM record identifiers

# balanced sample: min(20 + ID_SUM, Pass count, Fail count) per outcome
n_pass, n_fail = int((labels == -1).sum()), int((labels == 1).sum())
per_group = min(20 + ID_SUM, n_pass, n_fail)
rng = np.random.default_rng(RANDOM_SEED)
pass_ids = rng.choice(df.index[df["Outcome"] == "Pass"], size=per_group, replace=False)
fail_ids = rng.choice(df.index[df["Outcome"] == "Fail"], size=per_group, replace=False)
sample_ids = np.sort(np.concatenate([pass_ids, fail_ids]))

feat_pos = np.sort(rng.choice(usable, size=5, replace=False))
feats = [feature_names[i] for i in feat_pos]
full_medians = df[feats].median()                                     # medians from the FULL dataset
balanced = df.loc[sample_ids, feats + ["Result", "Outcome"]].copy()
balanced[feats] = balanced[feats].fillna(full_medians)

grouped = balanced.groupby("Outcome")[feats].agg(["count", "mean", "median", "std"])
print(grouped.round(4).to_string())

rows = []
for f in feats:
    p = balanced.loc[balanced["Outcome"] == "Pass", f]
    q = balanced.loc[balanced["Outcome"] == "Fail", f]
    abs_diff = abs(q.mean() - p.mean())
    pooled = np.sqrt(((len(p) - 1) * p.var(ddof=1) + (len(q) - 1) * q.var(ddof=1)) / (len(p) + len(q) - 2))
    smd = (q.mean() - p.mean()) / pooled if pooled > 0 else np.nan          # safe denominator
    rows.append({"Feature": f, "PassMean": p.mean(), "FailMean": q.mean(),
                 "AbsMeanDiff": abs_diff, "PooledSD": pooled, "StdMeanDiff": smd,
                 "AbsStdMeanDiff": abs(smd) if pooled > 0 else 0.0})
comp = pd.DataFrame(rows).sort_values("AbsStdMeanDiff", ascending=False).reset_index(drop=True)
comp.insert(0, "Rank", comp.index + 1)

# NumPy-Pandas statistics match
pandas_mean = balanced[feats[0]].mean()
numpy_mean = balanced[feats[0]].to_numpy().mean()
numpy_sd = balanced[feats[0]].to_numpy().std(ddof=1)
match = bool(np.isclose(pandas_mean, numpy_mean) and np.isclose(balanced[feats[0]].std(ddof=1), numpy_sd))

balanced.reset_index().to_csv(OUTPUT_DIR / "balanced_sample.csv", index=False)
comp.round(4).to_csv(OUTPUT_DIR / "feature_comparison.csv", index=False)

print("\n", comp.round(4).to_string(index=False))
print("\n=== Assessment Data ===")
print("Balanced Sample Size:", len(balanced), f"({per_group} Pass + {per_group} Fail)")
print("Selected Features:", feats)
print("Highest-Ranked Feature:", comp.loc[0, "Feature"])
print("Absolute Standardized Mean Difference:", round(float(comp.loc[0, "AbsStdMeanDiff"]), 4))
print("NumPy-Pandas Statistics Match:", match)
print("Required Outputs: balanced_sample.csv, feature_comparison.csv")

Records: 1567 | Features: 590 | Usable features: 474
Seed digit: 2 | ID sum: 23 | Random seed: 261932
        Feature_160                            Feature_167                        Feature_171                         Feature_196                        Feature_447                        
              count      mean median       std       count    mean median     std       count    mean  median     std       count    mean median     std       count    mean  median     std
Outcome                                                                                                                                                                                     
Fail             43  846.7907  494.0  971.3900          43  1.2930    1.3  0.4217          43  0.1247  0.1080  0.0627          43  7.6337   6.52  4.0642          43  0.2686  0.2668  0.0886
Pass             43  458.1860  387.0  216.5428          43  1.1953    1.2  0.4146          43  0.1225  0.1096  0.0607          43  7.8702   6.